# ECO346 Quiz 2 (Section B): your personal notebook

<a href="https://colab.research.google.com/github/rafi-amiruddin/DVTE/blob/main/data/quiz2/quiz2_sectionB.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

**What to do (in Google Colab, signed in with your CUI account)**

1. In the next cell, replace `FA24-BED-000` with **your own** registration number.
2. From the menu choose **Runtime → Run all**. If Colab warns that the notebook was not authored by Google, click **Run anyway**.
3. Read your results and answer the Google Form.

Change nothing else. If you change something by mistake, choose **Runtime → Restart session and run all**.
Every student gets different data, so your answers are yours alone. You do not need to save this notebook.

In [ ]:
REG = "FA24-BED-000"      # <-- change ONLY this line, e.g. "FA24-BED-047"

## 0. Setting up (just run it)

This cell loads your 3,000 persons from the course data file and works out your personal numbers.

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

try:
    plt.style.use("https://raw.githubusercontent.com/rafi-amiruddin/DVTE/main/style/dvte.mplstyle")
except Exception:
    pass
plt.rcParams.update({"figure.figsize": (9, 3.6), "font.size": 10})

REG = REG.strip().upper()
if not re.fullmatch(r"FA24-BED-\d{3}", REG):
    raise ValueError("Write your registration number exactly like FA24-BED-047, then run again.")

URL = "https://raw.githubusercontent.com/rafi-amiruddin/DVTE/main/data/quiz2/quiz2_data.csv"
data = pd.read_csv(URL)
me = data[data["reg"] == REG].reset_index(drop=True)
if me.empty:
    raise ValueError(f"{REG} is not in the quiz file. Check the number, or tell the instructor.")

n = int(REG[-3:])
b = n % 11 + 1              # histogram bin width (years)
h = (n // 2) % 10 + 1       # density bandwidth (years)
f = n % 7 + 1               # your figure for Q11-Q13
rng = np.random.default_rng(n)   # fixes the order of the versions shown to you

age = me["age"].to_numpy(float)
earn = me.loc[me["earn_annual"] > 0, "earn_annual"].to_numpy(float)

print(f"Registration number : {REG}")
print(f"Persons in your data: {len(me):,}   (earners: {len(earn):,})")
print(f"Your bin width b    : {b} years")
print(f"Your bandwidth h    : {h} years")
print(f"Your figure f       : Figure {f}")

## 1. Histogram of age (for Q1)

Below are **two versions** of the histogram of age in your data, both with bins of width *b* years starting at 0.
They differ in only one thing: what happens to a person whose age falls **exactly on the edge** between two bins.
Read the code line printed for each version.

In [ ]:
edges = np.arange(0, age.max() + b + 1, b)
versions = {
    "pd.cut(age, edges, right=False)": pd.cut(age, edges, right=False),
    "pd.cut(age, edges, right=True, include_lowest=True)": pd.cut(age, edges, right=True, include_lowest=True),
}
order = list(versions)
rng.shuffle(order)

fig, axes = plt.subplots(1, 2, sharey=True)
for ax, letter, call in zip(axes, "AB", order):
    counts = pd.Series(versions[call]).value_counts(sort=False).to_numpy()
    k = counts.argmax()
    ax.bar(edges[:-1], counts, width=b, align="edge", color="#3B6EA8", edgecolor="white")
    ax.set(title=f"Version {letter}", xlabel="Age (years)")
    print(f"Version {letter}:  {call}")
    print(f"           tallest bar: {counts[k]} persons, bin starting at age {edges[k]:g}\n")
axes[0].set_ylabel("Number of persons")
plt.tight_layout(); plt.show()

## 2. Density of age (for Q2)

Below are **three versions** of a Gaussian kernel density of age in your data.
Only one of them uses a bandwidth of exactly **2 × h years** (twice your h). Read the code line printed for each version,
and remember what `bw_method` measures in `scipy`'s `gaussian_kde`.

In [ ]:
grid = np.round(np.arange(0, 100.0001, 0.1), 1)
sd = age.std(ddof=1)
calls = {
    "gaussian_kde(age, bw_method = h / age.std())": h / sd,
    "gaussian_kde(age, bw_method = 2 * h / age.std())": 2 * h / sd,
    "gaussian_kde(age, bw_method = 2 * h)": 2 * h,
}
order = list(calls)
rng.shuffle(order)

fig, axes = plt.subplots(1, 3, sharey=True, figsize=(11, 3.4))
for ax, letter, call in zip(axes, "ABC", order):
    dens = gaussian_kde(age, bw_method=calls[call])(grid)
    ax.plot(grid, dens, color="#3B6EA8", lw=1.8)
    ax.set(title=f"Version {letter}", xlabel="Age (years)", xlim=(0, 100))
    print(f"Version {letter}:  {call}")
    print(f"           height of the curve at its highest point: {dens.max():.4f}\n")
axes[0].set_ylabel("Density")
plt.tight_layout(); plt.show()

## 3. Inequality of earnings (for Q3 and Q4)

Your earners are the persons in your data with annual earnings above zero.
The Lorenz curve and the table show what share of all their earnings goes to the poorest 10%, 20%, ... of them.

In [ ]:
e = np.sort(earn)
total = e.sum()
p = np.arange(0.1, 1.01, 0.1)
L = [e[: int(round(q * len(e)))].sum() / total for q in p]

table = pd.DataFrame({
    "Poorest share of earners": [f"{q:.0%}" for q in p],
    "Their share of all earnings (%)": [f"{100 * x:.1f}" for x in L],
})
try:
    display(table.style.hide(axis="index"))
except Exception:                      # older pandas
    print(table.to_string(index=False))

cum = np.concatenate([[0], np.cumsum(e) / total])
pop = np.linspace(0, 1, len(cum))
fig, ax = plt.subplots(figsize=(4.6, 4.4))
ax.plot([0, 1], [0, 1], ls="--", color="grey", lw=1)
ax.plot(pop, cum, color="#3B6EA8", lw=2)
ax.set(xlabel="Cumulative share of earners", ylabel="Cumulative share of earnings",
       xlim=(0, 1), ylim=(0, 1), aspect="equal", title="Your Lorenz curve")
plt.tight_layout(); plt.show()

## 4. Your figure (for Q11, Q12 and Q13)

The cell below draws **your** figure, number *f*, exactly as it was "published", from your own data.
Under it is the **claim** that the author made. The claim is wrong or unsupported because of how the figure was drawn.

Right after it, the notebook draws a **corrected** version of the same figure, without a title.
Use the first figure to answer Q11 and the corrected one for the numbers in Q12 and Q13.

In [ ]:
from matplotlib.ticker import FuncFormatter
rs = FuncFormatter(lambda v, _: f"{v/1e6:.1f}m" if v >= 1e6 else f"{v/1e3:.0f}k")
BLUE, ORANGE, GREEN = "#3B6EA8", "#F36619", "#2E7D32"
PROV = {1: "Khyber Pakhtunkhwa", 2: "Punjab", 3: "Sindh", 4: "Balochistan"}
logbins = np.logspace(4, 7.5, 30)
grid = np.round(np.arange(0, 100.0001, 0.1), 1)
band_edges = np.array([0, 1e5, 2e5, 3e5, 1e6])
e1m = earn[earn < 1e6]
men_age = me.loc[me.sex == 1, "age"].to_numpy(float)
women_age = me.loc[me.sex == 2, "age"].to_numpy(float)

def gkde(x, bw, g):
    z = (g[:, None] - x[None, :]) / bw
    return np.exp(-0.5 * z**2).sum(1) / (len(x) * bw * np.sqrt(2 * np.pi))

if f == 5:
    fig, axes = plt.subplots(1, 4, figsize=(11, 3.4))
    for ax, (code_, name) in zip(axes, PROV.items()):
        ax.hist(me.loc[me.prov_code == code_, "age"], bins=np.arange(0, 95, 5), color=BLUE, edgecolor="white")
        ax.set(title=name, xlabel="Age (years)")
    axes[0].set_ylabel("Number of persons")
    fig.suptitle(f"Figure {f}. Age of persons by province", x=0.01, ha="left", fontweight="bold", fontsize=11)
    claim = "Each province has about the same number of children under five."
else:
    fig, ax = plt.subplots(figsize=(7, 4))
    if f == 1:
        ax.hist(age, bins=np.arange(0, 120, 20), color=BLUE, edgecolor="white")
        ax.set(xlabel="Age (years)", ylabel="Number of persons")
        title = "Age of persons"
        claim = "The number of persons falls smoothly with age, so ages were reported accurately, with no bunching at round numbers."
    elif f == 2:
        cnt, _ = np.histogram(e1m, bins=band_edges)
        ax.bar(band_edges[:-1], cnt, width=np.diff(band_edges), align="edge", color=BLUE, edgecolor="white")
        ax.xaxis.set_major_formatter(rs)
        ax.set(xlabel="Annual earnings (Rs; earners above Rs 1m not shown)", ylabel="Number of earners")
        title = "Annual earnings of earners"
        claim = "The block from Rs 300k to Rs 1m is by far the biggest, so earners are most concentrated in that range."
    elif f == 3:
        ma = me.groupby("prov_code")["age"].mean().rename(PROV)
        lo, hi = ma.idxmin(), ma.idxmax()
        base = np.floor(ma.min()) - 1
        ax.bar(ma.index, ma.values, color=BLUE)
        ax.set_ylim(base, np.ceil(ma.max()) + 0.5)
        ax.set(ylabel="Mean age (years)")
        ratio = (ma.max() - base) / (ma.min() - base)
        title = "Mean age by province"
        claim = f"The average person in {hi} is about {ratio:.0f} times as old as the average person in {lo}."
    elif f == 4:
        sh = me["prov_code"].map(PROV).value_counts(normalize=True).reindex(PROV.values()) * 100
        ax.bar(sh.index, sh.values, color=BLUE)
        ax.set(ylabel="Percent of persons")
        title = "Where Pakistanis live"
        claim = f"Balochistan is home to about {sh['Balochistan']:.0f}% of Pakistan's population."
    elif f == 6:
        ax.hist(earn, bins=logbins, color=BLUE, edgecolor="white")
        ax.set_xscale("log"); ax.xaxis.set_major_formatter(rs)
        ax.set(xlabel="Annual earnings (Rs)", ylabel="Number of earners")
        title = "Annual earnings of earners"
        claim = "The histogram is roughly symmetric, so earnings are not skewed and the mean is a good typical value."
    else:
        ax.plot(grid, gkde(men_age, 1, grid), color=BLUE, lw=1.6, label="Men")
        ax.plot(grid, gkde(women_age, 8, grid), color=ORANGE, lw=2, label="Women")
        ax.set(xlabel="Age (years)", ylabel="Density", xlim=(0, 100)); ax.legend()
        title = "Age of men and women"
        claim = "Men's ages bunch at round numbers but women's do not, so women report their ages more accurately."
    ax.set_title(f"Figure {f}. {title}", loc="left", fontweight="bold", fontsize=11)

plt.tight_layout(); plt.show()
print(f"CLAIM (Figure {f}): {claim}")

**Corrected version of your figure** (no title; for Q12 and Q13)

In [ ]:
if f == 5:
    fig, axes = plt.subplots(1, 4, figsize=(11, 3.4), sharey=True)
    for ax, (code_, name) in zip(axes, PROV.items()):
        ax.hist(me.loc[me.prov_code == code_, "age"], bins=np.arange(0, 95, 5), color=BLUE, edgecolor="white")
        u5 = int((me.loc[me.prov_code == code_, "age"] < 5).sum())
        ax.set(title=f"{name} (under 5: {u5})", xlabel="Age (years)")
    axes[0].set_ylabel("Number of persons (same scale in every panel)")
else:
    fig, ax = plt.subplots(figsize=(7, 4))
    if f == 1:
        ax.hist(age, bins=np.arange(0, 101, 1), color=BLUE)
        ax.set(xlabel="Age (years), 1-year bins", ylabel="Number of persons")
    elif f == 2:
        cnt, _ = np.histogram(e1m, bins=band_edges)
        per100k = cnt / (np.diff(band_edges) / 1e5)
        ax.bar(band_edges[:-1], per100k, width=np.diff(band_edges), align="edge", color=BLUE, edgecolor="white")
        for x0, w, c, d in zip(band_edges[:-1], np.diff(band_edges), cnt, per100k):
            ax.text(x0 + w / 2, d, f"{c} earners", ha="center", va="bottom", fontsize=9)
        ax.xaxis.set_major_formatter(rs)
        ax.set(xlabel="Annual earnings (Rs; earners above Rs 1m not shown)", ylabel="Earners per Rs 100k of earnings")
    elif f == 3:
        ma = me.groupby("prov_code")["age"].mean().rename(PROV)
        ax.bar(ma.index, ma.values, color=BLUE)
        for i, v in enumerate(ma.values):
            ax.text(i, v, f"{v:.1f}", ha="center", va="bottom")
        ax.set(ylabel="Mean age (years)", ylim=(0, None))
    elif f == 4:
        w = me.groupby("prov_code")["hhweight"].sum().rename(PROV)
        sh = w / w.sum() * 100
        ax.bar(sh.index, sh.values, color=BLUE)
        for i, v in enumerate(sh.values):
            ax.text(i, v, f"{v:.1f}%", ha="center", va="bottom")
        ax.set(ylabel="Percent of population (weighted by hhweight)")
    elif f == 6:
        ax.hist(earn, bins=logbins, color=BLUE, edgecolor="white")
        for v, lab, col in [(np.mean(earn), "Mean", ORANGE), (np.median(earn), "Median", GREEN)]:
            ax.axvline(v, color=col, lw=2, label=f"{lab} = Rs {v/1e3:,.0f}k")
        ax.set_xscale("log"); ax.legend()
        ax.set(xlabel="Annual earnings (Rs, log scale)", ylabel="Number of earners")
    else:
        ax.plot(grid, gkde(men_age, 1, grid), color=BLUE, lw=1.4, label="Men")
        ax.plot(grid, gkde(women_age, 1, grid), color=ORANGE, lw=1.4, label="Women")
        ax.set(xlabel="Age (years)", ylabel="Density (bandwidth 1 year for both)", xlim=(0, 100)); ax.legend()
plt.tight_layout(); plt.show()

---
You are done with the notebook. Go back to the Google Form and answer the questions.
Keep this notebook open until you have submitted the form.